# Does the advantage hold past sixteen hospitals?

Section 5.9 shows the ECE advantage flat with per-client data held at
1,000 images: 1.46x, 1.48x, 1.42x, 1.42x at K = 4, 8, 12, 16. The paper
says plainly that it has not been tested beyond sixteen sites, which is
an open limitation a reviewer can point at.

This adds K = 20 and K = 24 at the same 25% corrupted fraction and the
same 1,000 images per client. If the ratio stays near 1.42 the
limitation closes. If it bends, that is the size limit the paper has
been looking for, and Section 5.9 gains a real boundary instead of an
untested edge.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus

In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
import traceback
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456,7,13,21,99,2024,31337,555]
# 25% corrupted, per-client data fixed at 1000 -- the corpus grows with K,
# which is how federations actually grow (new hospitals bring their data).
GRID=[(20,5),(24,6)]

done=fail=0
for K,nn in GRID:
    for s in SEEDS:
        common=dict(dataset='pathmnist', num_classes=9,
                    num_clients=K, num_noisy=nn, noisy_client=K-1,
                    samples_per_client=1000, seed=s)
        for name, kw in ((f'fixn_fedavg_k{K}_n{nn}_s{s}', dict(aggregator='fedavg')),
                         (f'fixn_aha_k{K}_n{nn}_s{s}',
                          dict(aggregator='adaptive', alpha=1.0,
                               beta=4.0, gamma=4.0))):
            try:
                run_one(OUT_DIR/f'{name}.json', **kw, **common)
                done+=1
            except Exception:
                fail+=1
                print(f'FAILED {name}'); traceback.print_exc()
    print(f'K={K} done ({done} ok, {fail} failed)', flush=True)
print(f'TOTAL {done} ok, {fail} failed')


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def by_seed(pat, key='ece'):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

print('25% corrupted, 1000 images per client')
print('  K    AHA-Fed   FedAvg    ratio    p        n')
seen={}
for K,nn in [(4,1),(8,2),(12,3),(16,4),(20,5),(24,6)]:
    if K==4:
        a=by_seed('runs/paper/real/multi_aha_n1_s*.json')
        v=by_seed('runs/paper/real/multi_fedavg_n1_s*.json')
    else:
        a=by_seed(f'runs/paper/real/fixn_aha_k{K}_n{nn}_s*.json')
        v=by_seed(f'runs/paper/real/fixn_fedavg_k{K}_n{nn}_s*.json')
    c=sorted(set(a)&set(v))
    if not c: continue
    x=[a[k] for k in c]; y=[v[k] for k in c]
    p=wilcoxon(x,y).pvalue if len(c)>1 else float('nan')
    seen[K]=st.mean(y)/st.mean(x)
    print(f'  {K:2d}   {st.mean(x):.4f}   {st.mean(y):.4f}   '
          f'{seen[K]:.2f}x   p={p:.4f}  {len(c)}')

if 20 in seen and 24 in seen:
    print()
    if min(seen[20], seen[24]) > 1.25:
        print('=> the advantage holds past sixteen sites; the limitation closes')
    else:
        print('=> it bends here -- this is the size limit, and 5.9 must say so')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/k24_results.zip runs/paper/real
!ls -lh /kaggle/working/k24_results.zip
